In [41]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
전처리 과정 테스트 스크립트 - 일반 코드 버전
- FMP 데이터의 NaN 값 확인
- DB 보완 후 NaN 값 확인
- 각 단계별 데이터 상태 리포트
"""

import requests
import pandas as pd
import numpy as np
import calendar
import time

from tqdm import tqdm
import warnings
from sqlalchemy import create_engine
from DATA.stock_invest_function import *
from datetime import datetime
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore')

# 설정값들
ticker = 'AMAT'

hs_code = '841191'

api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

start_date = '2013-01-01'

# 유틸리티 함수들
def convert_to_month_end(date_str):
    try:
        # 문자열/타입 혼용 안전 변환
        date_obj = pd.to_datetime(date_str)
        if pd.isna(date_obj):
            return None

        y, m, d = date_obj.year, date_obj.month, date_obj.day

        # 1~5일 → 전달 말일
        if 1 <= d <= 5:
            if m == 1:
                prev_y, prev_m = y - 1, 12
            else:
                prev_y, prev_m = y, m - 1
            last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
            return datetime(prev_y, prev_m, last_day_prev)

        # 그 외 → 해당월 말일
        last_day_cur = calendar.monthrange(y, m)[1]
        return datetime(y, m, last_day_cur)

    except Exception:
        return None

def create_monthly_end_dataframe(start_date):
    """
    start_date부터 이번달 전달까지 매월 말 기준으로
    날짜와 월 순서 더미 변수가 포함된 데이터프레임을 생성
    """
    start = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.now().replace(day=1) - relativedelta(days=1)  # 이번달 전달 말일

    monthly_ends = []
    current_date = start.replace(day=1)

    while current_date <= end:
        next_month = current_date + relativedelta(months=1)
        month_end = next_month - relativedelta(days=1)

        if month_end <= end:
            monthly_ends.append(month_end)

        current_date = next_month

    return pd.DataFrame({
        'date_month_end': monthly_ends,
        'month_dummy': range(1, len(monthly_ends) + 1)
    })

def add_revenue_ttm(df):
    """Add TTM (Trailing Twelve Months) column to quarterly revenue data"""
    df_copy = df.copy()
    df_copy = df_copy.sort_values(['ticker', 'date'])
    ttm_values = []
    for ticker in df_copy['ticker'].unique():
        ticker_data = df_copy[df_copy['ticker'] == ticker].copy()
        ticker_data = ticker_data.sort_values('date')
        ticker_data['revenue_ttm'] = ticker_data['revenue'].rolling(window=4, min_periods=1).sum()
        ttm_values.extend(ticker_data['revenue_ttm'].tolist())
    df_copy['revenue_ttm'] = ttm_values
    return df_copy


def fetch_revenue_data(ticker, api_key):
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"
        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"
        if not data:
            return None, "데이터 없음"
        return data, None
    except Exception as e:
        return None, f"오류: {str(e)}"

def fetch_market_data_yearly(ticker, api_key, start_year=2010):
    all_data = []
    current_year = datetime.now().year
    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"
        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
            time.sleep(0.3)
        except Exception as e:
            continue
    return all_data if all_data else None, None

def process_daily_to_monthly_market_data(daily_data, ticker):
    if not daily_data:
        return pd.DataFrame()
    df = pd.DataFrame(daily_data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_data = []
    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]
        last_day_data = month_data.loc[month_data['date'].idxmax()]
        monthly_data.append({
            'ticker': ticker,
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
        })
    return pd.DataFrame(monthly_data)

def fetch_db_revenue_data(ticker, db_info, end_date='2025-08-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, saleq
        FROM US_fundq
        WHERE ticker = '{ticker}'
        AND saleq IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['revenue_billions'] = df['saleq'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'revenue_billions']]
    except Exception as e:
        return pd.DataFrame()

def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['market_cap_billions'] = df['me'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
    except Exception as e:
        return pd.DataFrame()

# ==============================================
# Export Data Collection Functions
# ==============================================

def get_hs_data(hs_code_6d, db_info):
    """Extract trade data by HS Code (2013-2024)"""
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT * FROM us_trade_monthly_data_with_forecast
        WHERE hs_code_6d = '{hs_code_6d}'
        AND date >= '2013-01-01'
        AND date <= '2026-12-31'
        ORDER BY date DESC
        """
        df = pd.read_sql(query, engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
        return df
    except Exception:
        return pd.DataFrame()


def get_latest_input_date_data(df):
    """Extract data with the latest input_date only"""
    if 'input_date' not in df.columns:
        return pd.DataFrame()
    df_copy = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(df_copy['input_date']):
        df_copy['input_date'] = pd.to_datetime(df_copy['input_date'])
    latest_date = df_copy['input_date'].max()
    latest_data = df_copy[df_copy['input_date'] == latest_date].copy()
    return latest_data


def collect_export_data(hs_code, db_info):
    """Collect export data (2013-2024)"""
    export_df = get_hs_data(hs_code, db_info)
    if export_df.empty:
        return pd.DataFrame()
    latest_export_data = get_latest_input_date_data(export_df)
    latest_export_data = latest_export_data.sort_values('date').reset_index(drop=True)
    if not latest_export_data.empty:
        latest_export_data['date'] = pd.to_datetime(latest_export_data['date'])
        latest_export_data['date_month_end'] = latest_export_data['date'].apply(convert_to_month_end)
    return latest_export_data



# ==============================================
# Data Merge and PSR Calculation Functions
# ==============================================

def calculate_enhanced_ttm_and_psr(merged_data):
    """Calculate enhanced TTM and PSR"""
    df = merged_data.copy()
    df = df.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    # Calculate TTM from quarterly revenue
    df['revenue_ttm'] = df.groupby('ticker')['revenue_billions'].rolling(window=4, min_periods=1).sum().reset_index(0,
                                                                                                                    drop=True)
    df['revenue_ttm_billions'] = df['revenue_ttm']

    # Apply 2-month shift
    df['revenue_ttm_shift'] = df.groupby('ticker')['revenue_ttm_billions'].shift(2)

    # Calculate PSR
    df['PSR_ttm'] = df['market_cap_billions'] / df['revenue_ttm_shift']

    # Handle infinite values
    df['PSR_ttm'] = df['PSR_ttm'].replace([np.inf, -np.inf], np.nan)

    return df


def calculate_export_yoy_growth(df):
    """Calculate YoY growth rate for export data"""
    if 'expDlr' not in df.columns or df['expDlr'].isna().all():
        df['expDlr_yoy'] = pd.NA
        return df

    df = df.sort_values('date_month_end').reset_index(drop=True)
    df['expDlr_yoy'] = df['expDlr'].pct_change(periods=12) * 100  # YoY growth rate (%)

    return df


def merge_with_export_data(merged_data, export_data):
    """Merge final data with export data - preserve export forecasts"""
    if export_data.empty:
        merged_data['hs_code_6d'] = None
        merged_data['expDlr'] = pd.NA
        merged_data['expDlr_yoy'] = pd.NA
        return merged_data

    # Extract required columns from export data
    export_subset = export_data[['date_month_end', 'hs_code_6d', 'expDlr']].copy()

    # Calculate YoY growth rate
    export_subset = calculate_export_yoy_growth(export_subset)

    # Merge data (outer join to preserve all data)
    final_data = pd.merge(merged_data, export_subset, on='date_month_end', how='outer')

    # Sort by date
    final_data = final_data.sort_values('date_month_end').reset_index(drop=True)

    # Fill ticker NaN values with ffill
    if 'ticker' in final_data.columns:
        final_data['ticker'] = final_data['ticker'].ffill()
        # Apply bfill for cases where first row is NaN
        final_data['ticker'] = final_data['ticker'].bfill()

    return final_data

In [42]:
print("=" * 80)
print("전처리 과정 테스트 시작")
print(f"대상 종목: {ticker}")
print("=" * 80)

# 1. FMP 매출 데이터 수집
print("\n1. FMP 매출 데이터 수집 중...")
revenue_data, error = fetch_revenue_data(ticker, api_key)

if revenue_data is None:
    print(f"ERROR: FMP 매출 데이터 수집 실패 - {error}")
    exit()

all_revenue_data = []
for item in revenue_data:
    all_revenue_data.append({
        'ticker': ticker,
        'date': item.get('date', ''),
        'calendar_year': item.get('calendarYear', ''),
        'period': item.get('period', ''),
        'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
        'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
    })

fmp_revenue_df = pd.DataFrame(all_revenue_data)
fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)
fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)
print(f"FMP 매출 데이터: {len(fmp_revenue_df)}건")


# 2. DB 매출 데이터 가져오기
db_revenue_df = fetch_db_revenue_data(ticker, db_info)

# 3. FMP 시가총액 데이터 수집
print("2. FMP 시가총액 데이터 수집 중...")
market_data, error = fetch_market_data_yearly(ticker, api_key, start_year=2010)

if not market_data:
    print("ERROR: FMP 시가총액 데이터 수집 실패")
    exit()

fmp_market_df = process_daily_to_monthly_market_data(market_data, ticker)
fmp_market_df['date_month_end'] = fmp_market_df['date'].apply(convert_to_month_end)

print(f"FMP 시가총액 데이터: {len(fmp_market_df)}건")

# 4. FMP 원본 데이터의 NaN 값 확인
print("\n3. FMP 원본 데이터 NaN 분석")
print("-" * 50)

base_df = create_monthly_end_dataframe(start_date)

mereged_rev_data = pd.merge(base_df, fmp_revenue_df, on = 'date_month_end', how='outer')
mereged_rev_data.ffill(limit=2, inplace=True)
db_revenue_df.drop_duplicates(subset=['date_month_end'], inplace=True)

# 먼저 revenue 보충: date_month_end 기준 merge
mereged_rev_data = mereged_rev_data.merge(
    db_revenue_df[['date_month_end', 'revenue_billions']],
    on='date_month_end',
    how='left',
    suffixes=('', '_db')
)

# NaN 값 보충: revenue_billions가 NaN이면 db_revenue_df 값으로 채움
mereged_rev_data['revenue_billions'] = mereged_rev_data['revenue_billions'].fillna(
    mereged_rev_data['revenue_billions_db']
)

# 보조 컬럼 제거
# mereged_rev_data.drop(columns=['revenue_billions_db'], inplace=True)

mereged_rev_data[['ticker', 'calendar_year', 'period']] = mereged_rev_data[['ticker', 'calendar_year', 'period']].ffill()

merged_revenue_df = mereged_rev_data[['date_month_end', 'month_dummy', 'ticker', 'calendar_year', 'period', 'revenue_billions', 'revenue_billions_db']].copy()
# month_dummy가 NaN인 행 제거
merged_revenue_df= merged_revenue_df.dropna(subset=['month_dummy']).reset_index(drop=True)

전처리 과정 테스트 시작
대상 종목: AMAT

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 159건
2. FMP 시가총액 데이터 수집 중...
FMP 시가총액 데이터: 189건

3. FMP 원본 데이터 NaN 분석
--------------------------------------------------


In [43]:
db_market_df =  fetch_db_market_data(ticker, db_info)

# 1. db_market_df 컬럼 이름 변경
db_market_df_renamed = db_market_df.rename(
    columns={'market_cap_billions': 'market_cap_billions_from_db'}
)

# 2. 두 데이터프레임 merge (date_month_end 기준)
merged_market_df = fmp_market_df.merge(
    db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
    on='date_month_end',
    how='outer'   # outer join으로 모든 데이터 보존
)

# 3. NaN 값 보충: market_cap_billions NaN이면 from_db 값으로 채움
merged_market_df['market_cap_billions'] = merged_market_df['market_cap_billions'].fillna(
    merged_market_df['market_cap_billions_from_db']
)

merged_market_df = merged_market_df.dropna(subset=['ticker'])
merged_market_df = merged_market_df.drop_duplicates(subset=['date_month_end'], keep='first')

enhanced_merged_df = pd.merge(merged_revenue_df, merged_market_df[['date_month_end', 'market_cap_billions']], on='date_month_end', how='inner')
enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(enhanced_merged_df)

In [44]:
export_data = collect_export_data(hs_code, db_info)

In [45]:
finaal_df = merge_with_export_data(enhanced_merged_df_with_ttm, export_data)

In [46]:
exog_start_date = '2014-01-01'
finaal_df_resize = finaal_df[finaal_df['date_month_end'] >= exog_start_date]
finaal_df_resize['date_month_end'] = pd.to_datetime(finaal_df_resize['date_month_end'])


In [113]:
import importlib
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)
import importlib
import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)
import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)
from DATA.us_ets_forecast import *
import importlib
import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)

<module 'DATA.us_est_forecast_v2' from 'C:\\Users\\MetaM\\PycharmProjects\\stock_forecast\\DATA\\us_est_forecast_v2.py'>

In [114]:
sarima_df, sarima_results = sarima.run_sarima_prediction(
    df=finaal_df_resize,
    ticker="AMAT",
    forecast_quarters=4,
    start_date_revenue="2025-07-31",   # 매출 예측 시작일
    start_date_psr="2025-8-31",       # PSR 예측 시작일
    exog_col="expDlr_yoy"              # 외생변수 (None이면 미포함)
)

In [118]:
sarima_df[['revenue_billions_sarima_noexog', 'revenue_billions_sarima_exog']] = sarima_df[['revenue_billions_sarima_noexog', 'revenue_billions_sarima_exog']].ffill(limit=2)


In [119]:
sarima_df.tail(24)

,date_month_end,month_dummy,ticker,calendar_year,period,revenue_billions,revenue_billions_db,market_cap_billions,revenue_ttm,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr,expDlr_yoy,revenue_billions_sarima_noexog,revenue_billions_sarima_exog,PSR_sarima_forecast_noexog,PSR_sarima_forecast_exog
129,2024-10-31,142.0,AMAT,2024,Q4,7.04,7.045,147.81,27.38,27.38,26.86,5.502978,841191,247061000.0,24.201186,7.040000,7.040000,5.502978,5.502978
130,2024-11-30,143.0,AMAT,2024,Q4,7.04,7.045,142.21,27.64,27.64,26.99,5.268989,841191,255786000.0,63.795290,7.040000,7.040000,5.268989,5.268989
131,2024-12-31,144.0,AMAT,2024,Q4,7.04,7.045,132.38,27.90,27.90,27.38,4.834916,841191,213521000.0,-11.345056,7.040000,7.040000,4.834916,4.834916
132,2025-01-31,145.0,AMAT,2025,Q1,7.17,7.166,146.26,28.29,28.29,27.64,5.291606,841191,248374000.0,14.966673,7.170000,7.170000,5.291606,5.291606
133,2025-02-28,146.0,AMAT,2025,Q1,7.17,7.166,128.19,28.42,28.42,27.90,4.594624,841191,226273000.0,11.710515,7.170000,7.170000,4.594624,4.594624
134,2025-03-31,147.0,AMAT,2025,Q1,7.17,7.166,117.69,28.55,28.55,28.29,4.160127,841191,263404000.0,19.437013,7.170000,7.170000,4.160127,4.160127
135,2025-04-30,148.0,AMAT,2025,Q2,7.10,7.100,122.23,28.61,28.61,28.42,4.300844,841191,246104000.0,26.244082,7.100000,7.100000,4.300844,4.300844
136,2025-05-31,149.0,AMAT,2025,Q2,7.10,7.100,127.12,28.54,28.54,28.55,4.452539,841191,278946000.0,10.581836,7.100000,7.100000,4.452539,4.452539
137,2025-06-30,150.0,AMAT,2025,Q2,7.10,7.100,148.47,28.47,28.47,28.61,5.189444,841191,271430000.0,4.608590,7.100000,7.100000,5.189444,5.189444
138,2025-07-31,151.0,AMAT,2025,Q3,7.30,7.302,146.03,28.60,28.60,28.54,5.116678,841191,237667000.0,33.921045,7.300000,7.300000,5.116678,5.116678


In [108]:
## ---------------------------------------------------------
# 2) 예측 실행
# ---------------------------------------------------------
lstm_df, model_results = lstm_v2.run_lstm_prediction(
    df=finaal_df_resize,
    ticker='AMAT',
    prediction_quarters=4,
    start_date="2025-09-30"
)


In [109]:
lstm_df.tail(24)

,date_month_end,month_dummy,ticker,calendar_year,period,revenue_billions,revenue_billions_db,market_cap_billions,revenue_ttm,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr,expDlr_yoy,revenue_billions_lstm_forecast,PSR_lstm_forecast
129,2024-10-31,142.0,AMAT,2024,Q4,7.04,7.045,147.81,27.38,27.38,26.86,5.502978,841191,247061000.0,24.201186,7.040000,5.502978
130,2024-11-30,143.0,AMAT,2024,Q4,7.04,7.045,142.21,27.64,27.64,26.99,5.268989,841191,255786000.0,63.795290,7.040000,5.268989
131,2024-12-31,144.0,AMAT,2024,Q4,7.04,7.045,132.38,27.90,27.90,27.38,4.834916,841191,213521000.0,-11.345056,7.040000,4.834916
132,2025-01-31,145.0,AMAT,2025,Q1,7.17,7.166,146.26,28.29,28.29,27.64,5.291606,841191,248374000.0,14.966673,7.170000,5.291606
133,2025-02-28,146.0,AMAT,2025,Q1,7.17,7.166,128.19,28.42,28.42,27.90,4.594624,841191,226273000.0,11.710515,7.170000,4.594624
134,2025-03-31,147.0,AMAT,2025,Q1,7.17,7.166,117.69,28.55,28.55,28.29,4.160127,841191,263404000.0,19.437013,7.170000,4.160127
135,2025-04-30,148.0,AMAT,2025,Q2,7.10,7.100,122.23,28.61,28.61,28.42,4.300844,841191,246104000.0,26.244082,7.100000,4.300844
136,2025-05-31,149.0,AMAT,2025,Q2,7.10,7.100,127.12,28.54,28.54,28.55,4.452539,841191,278946000.0,10.581836,7.100000,4.452539
137,2025-06-30,150.0,AMAT,2025,Q2,7.10,7.100,148.47,28.47,28.47,28.61,5.189444,841191,271430000.0,4.608590,7.100000,5.189444
138,2025-07-31,151.0,AMAT,2025,Q3,7.30,7.302,146.03,28.60,28.60,28.54,5.116678,841191,237667000.0,33.921045,7.300000,5.116678


In [110]:
prophet_df, prophet_results = prophet_v3.run_prophet_prediction_v3(
    df=finaal_df_resize,
    ticker="AMAT",
    prediction_quarters=4,
    start_date_revenue="2025-07-31",   # Revenue 예측 시작일
    start_date_psr="2025-08-31",       # PSR 예측 시작일
    exog_col="expDlr_yoy"              # 외생변수 (None 주면 미사용)
)

18:29:57 - cmdstanpy - INFO - Chain [1] start processing
18:29:57 - cmdstanpy - INFO - Chain [1] done processing
18:29:57 - cmdstanpy - INFO - Chain [1] start processing
18:29:57 - cmdstanpy - INFO - Chain [1] done processing
18:29:58 - cmdstanpy - INFO - Chain [1] start processing
18:29:58 - cmdstanpy - INFO - Chain [1] done processing
18:29:58 - cmdstanpy - INFO - Chain [1] start processing
18:29:58 - cmdstanpy - INFO - Chain [1] done processing


In [111]:
prophet_df.tail(24)

,date_month_end,month_dummy,ticker,calendar_year,period,revenue_billions,revenue_billions_db,market_cap_billions,revenue_ttm,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr,expDlr_yoy,revenue_billions_prophet_forecast_noexog,revenue_billions_prophet_forecast_exog,PSR_prophet_forecast_noexog,PSR_prophet_forecast_exog
129,2024-10-31,142.0,AMAT,2024,Q4,7.04,7.045,147.81,27.38,27.38,26.86,5.502978,841191,247061000.0,24.201186,7.040000,7.040000,5.502978,5.502978
130,2024-11-30,143.0,AMAT,2024,Q4,7.04,7.045,142.21,27.64,27.64,26.99,5.268989,841191,255786000.0,63.795290,7.040000,7.040000,5.268989,5.268989
131,2024-12-31,144.0,AMAT,2024,Q4,7.04,7.045,132.38,27.90,27.90,27.38,4.834916,841191,213521000.0,-11.345056,7.040000,7.040000,4.834916,4.834916
132,2025-01-31,145.0,AMAT,2025,Q1,7.17,7.166,146.26,28.29,28.29,27.64,5.291606,841191,248374000.0,14.966673,7.170000,7.170000,5.291606,5.291606
133,2025-02-28,146.0,AMAT,2025,Q1,7.17,7.166,128.19,28.42,28.42,27.90,4.594624,841191,226273000.0,11.710515,7.170000,7.170000,4.594624,4.594624
134,2025-03-31,147.0,AMAT,2025,Q1,7.17,7.166,117.69,28.55,28.55,28.29,4.160127,841191,263404000.0,19.437013,7.170000,7.170000,4.160127,4.160127
135,2025-04-30,148.0,AMAT,2025,Q2,7.10,7.100,122.23,28.61,28.61,28.42,4.300844,841191,246104000.0,26.244082,7.100000,7.100000,4.300844,4.300844
136,2025-05-31,149.0,AMAT,2025,Q2,7.10,7.100,127.12,28.54,28.54,28.55,4.452539,841191,278946000.0,10.581836,7.100000,7.100000,4.452539,4.452539
137,2025-06-30,150.0,AMAT,2025,Q2,7.10,7.100,148.47,28.47,28.47,28.61,5.189444,841191,271430000.0,4.608590,7.100000,7.100000,5.189444,5.189444
138,2025-07-31,151.0,AMAT,2025,Q3,7.30,7.302,146.03,28.60,28.60,28.54,5.116678,841191,237667000.0,33.921045,7.300000,7.300000,5.116678,5.116678


In [112]:
es_df, es_results = esmod.run_es_prediction_v1(
    df=finaal_df_resize,
    ticker="AMAT",
    prediction_quarters=4,            # 매출 4개 분기(=12개월)
    start_date_revenue="2025-07-31",  # 매출 예측 시작일
    start_date_psr="2025-08-31"       # PSR 예측 시작일 (서로 다르게 지정 가능)
)

# 결과 확인
es_df[['date_month_end','revenue_billions','revenue_billions_es_forecast',
       'PSR_ttm','PSR_es_forecast']].tail(24)

,date_month_end,revenue_billions,revenue_billions_es_forecast,PSR_ttm,PSR_es_forecast
129,2024-10-31,7.04,7.040000,5.502978,5.502978
130,2024-11-30,7.04,7.040000,5.268989,5.268989
131,2024-12-31,7.04,7.040000,4.834916,4.834916
132,2025-01-31,7.17,7.170000,5.291606,5.291606
133,2025-02-28,7.17,7.170000,4.594624,4.594624
134,2025-03-31,7.17,7.170000,4.160127,4.160127
135,2025-04-30,7.10,7.100000,4.300844,4.300844
136,2025-05-31,7.10,7.100000,4.452539,4.452539
137,2025-06-30,7.10,7.100000,5.189444,5.189444
138,2025-07-31,7.30,7.300000,5.116678,5.116678
